# Next Seodang OpenAI + Langfuse Agent Demo

이 노트북은 발표에서 바로 시연할 수 있도록 만든 **실전형 Colab 실습 노트북**입니다.

이 노트북이 보여주는 것:

1. **회의 메모 정리 Agent**
2. **토론 질문 생성 Agent**
3. **발표 피드백 Agent**
4. **Langfuse Trace를 통해 어떤 함수가 실제로 호출됐는지 확인**

중요한 교육 포인트:

- OpenAI API로 실제 Agent를 실행한다.
- Langfuse를 붙여서 `요청 -> 모델 호출 -> 함수 호출 -> 최종 응답` 흐름을 Trace로 본다.
- 일부 도구 함수 내부도 LLM을 다시 호출해서, 발표 시연용으로 충분히 납득 가능한 품질을 만든다.

즉, 이 노트북은 단순한 toy demo가 아니라,
**작은 Agent 구조를 실제 품질과 실제 Trace까지 포함해 보여주는 발표용 예제**다.

## PPT와 연결해서 보면 좋은 포인트

이 노트북은 발표 자료의 다음 장표와 직접 연결됩니다.

- `Slide 43~48`: Functions / MCP / A2A
- `Slide 49~54`: 프로젝트 기반 AI 협업
- `Slide 55~58`: Colab 실습
- `Slide 60`: Skill Setting과 품질 검수 루프

발표 중에는 아래처럼 연결해서 설명하면 좋습니다.

- **Functions**: 모델이 함수를 선택하는 장면
- **Trace**: 그 함수가 실제로 어떻게 불렸는지 보는 장면
- **AI Native 운영법**: 결과를 검수하고 축적하는 장면

In [ ]:
# OpenAI SDK와 Langfuse SDK를 설치합니다.
#
# - openai: 실제 모델 호출
# - langfuse: Trace / observation / OpenAI 호출 관측
%pip install -q --upgrade openai langfuse

## 1. API Key 설정

이 노트북은 **OpenAI API Key**와 **Langfuse 키**를 함께 사용합니다.

권장 방식은 Colab `Secrets`에 아래 이름으로 저장하는 것입니다.

- `OPENAI_API_KEY`
- `LANGFUSE_SECRET_KEY`
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_BASE_URL`

주의:

- 키를 노트북에 직접 저장하지 않습니다.
- 발표가 끝난 뒤에는 데모용 키를 회전하는 것이 안전합니다.

In [ ]:
# Colab Secrets / 환경변수 / 직접 입력 순서로 키를 불러옵니다.

import os
from getpass import getpass


def get_secret(name: str, prompt_if_missing: bool = False) -> str | None:
    value = os.getenv(name)
    try:
        from google.colab import userdata
        value = value or userdata.get(name)
    except Exception:
        pass
    if not value and prompt_if_missing:
        value = getpass(f"Enter {name}: ")
    if value:
        os.environ[name] = value
    return value


OPENAI_API_KEY = get_secret("OPENAI_API_KEY", prompt_if_missing=True)
LANGFUSE_SECRET_KEY = get_secret("LANGFUSE_SECRET_KEY", prompt_if_missing=False)
LANGFUSE_PUBLIC_KEY = get_secret("LANGFUSE_PUBLIC_KEY", prompt_if_missing=False)
LANGFUSE_BASE_URL = get_secret("LANGFUSE_BASE_URL", prompt_if_missing=False) or "https://cloud.langfuse.com"

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("LANGFUSE tracing enabled:", bool(LANGFUSE_SECRET_KEY and LANGFUSE_PUBLIC_KEY))
print("LANGFUSE_BASE_URL:", LANGFUSE_BASE_URL)

In [ ]:
# ------------------------------------------------------------------
# Imports & client setup
# ------------------------------------------------------------------
#
# 여기서는 일반 OpenAI 클라이언트 대신,
# Langfuse가 감싼 OpenAI wrapper를 사용합니다.
# 이렇게 하면 OpenAI 호출이 자동으로 Langfuse에 기록됩니다.

import json
from pathlib import Path
from typing import Any

from langfuse import get_client, observe, propagate_attributes
from langfuse.openai import openai

# Langfuse OpenAI wrapper는 module-level openai 사용 방식을 따릅니다.
openai.api_key = OPENAI_API_KEY

MODEL = "gpt-4.1-mini"
TOOL_MODEL = "gpt-4.1-mini"

# Langfuse client는 flush() 용도로 씁니다.
langfuse = get_client()

TRACE_ENABLED = bool(LANGFUSE_SECRET_KEY and LANGFUSE_PUBLIC_KEY)
DEMO_SESSION_ID = "next-seodang-agent-demo"
DEMO_USER_ID = "lecture-presenter"

print("main model =", MODEL)
print("tool model =", TOOL_MODEL)
print("trace enabled =", TRACE_ENABLED)

## 2. Langfuse Trace에서 무엇을 볼 수 있나

이 노트북은 Trace에서 아래를 보이도록 설계했습니다.

1. **Agent 실행 전체 span**
2. **모델의 OpenAI 호출**
3. **각 tool 함수 실행 observation**
4. **도구 함수 내부에서 다시 일어난 LLM 호출**

즉, Trace를 열면 단순히 답변 결과만 보는 것이 아니라,
`어떤 요청에서 어떤 함수가 실제로 쓰였는지`를 구분해서 볼 수 있습니다.

## 3. 예시 문서 불러오기

발표 자료와 연결해서 보기 좋게, 긴 예시 문서를 따로 준비했습니다.

- `examples/data/seminar_meeting_notes_long.md`
- `examples/data/discussion_material_ai_native.md`
- `examples/data/presentation_draft_ai_native.md`

각 문서에는 **Trace에서 어떤 tool이 호출되는 것이 기대되는지**도 적어두었습니다.

In [ ]:
def safe_read_text(path: str, fallback: str) -> str:
    file_path = Path(path)
    if file_path.exists():
        return file_path.read_text(encoding="utf-8")
    return fallback


meeting_notes = safe_read_text(
    "examples/data/seminar_meeting_notes_long.md",
    "발표 자료를 정리하고, 액션아이템과 리허설 일정 관련 항목을 뽑아야 한다.",
)

discussion_material = safe_read_text(
    "examples/data/discussion_material_ai_native.md",
    "AI Native 시대의 커리어에 대한 토론 자료 예시",
)

presentation_draft = safe_read_text(
    "examples/data/presentation_draft_ai_native.md",
    "AI Native는 AI를 많이 쓰는 사람이 아니라, AI를 붙여 자신의 사고와 실행을 확장하는 사람이다.",
)

print("meeting_notes length:", len(meeting_notes))
print("discussion_material length:", len(discussion_material))
print("presentation_draft length:", len(presentation_draft))

## 4. 도구 함수 설계

오늘은 발표 품질을 위해 **일부 도구 함수 내부도 LLM을 다시 호출**합니다.

이 점은 꼭 설명해야 합니다.

- 보통은 tool 함수가 단순한 로컬 함수인 편이 더 관리하기 쉽습니다.
- 하지만 오늘은 **결과 품질을 보여주는 시연**이 목적이므로,
  요약/질문 생성/발표 피드백 도구가 내부적으로 `TOOL_MODEL`을 다시 호출합니다.
- 이 구조 덕분에 Langfuse Trace에서는
  `agent -> tool -> nested model call` 구조를 확인할 수 있습니다.

In [ ]:
@observe(name="tool_model_call", as_type="generation")
def llm_text(system_prompt: str, user_prompt: str, model: str = TOOL_MODEL, generation_name: str = "tool-model-call") -> str:
    """
    도구 함수 내부에서 LLM을 한 번 더 호출할 때 사용하는 공통 헬퍼입니다.
    Langfuse Trace에서는 generation observation으로 잡히게 됩니다.
    """
    response = openai.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        name=generation_name,
        metadata={"layer": "tool_internal_llm", "model_role": "tool"},
    )
    return response.output_text.strip()


@observe(name="summarize_notes", as_type="tool")
def summarize_notes(notes: str) -> dict[str, Any]:
    prompt = f"""
    아래 회의 메모 또는 문서를 읽고 다음 형식으로 정리하라.

    1. 한 문단 요약
    2. 핵심 포인트 5개
    3. 남은 과제 3개
    4. 발표자가 꼭 말해야 할 핵심 메시지 1줄

    [원문]
    {notes}
    """
    text = llm_text(
        system_prompt="너는 회의 메모와 문서를 구조적으로 압축하는 편집자다. 짧지만 밀도 있게 정리하라.",
        user_prompt=prompt,
        generation_name="tool-summarize-notes-llm",
    )
    return {"summary_markdown": text}


@observe(name="extract_action_items", as_type="tool")
def extract_action_items(notes: str) -> dict[str, Any]:
    prompt = f"""
    아래 메모를 읽고 액션아이템만 추출하라.

    형식:
    - 해야 할 일
    - 왜 중요한지 짧은 설명
    - 우선순위(상/중/하)

    [메모]
    {notes}
    """
    text = llm_text(
        system_prompt="너는 프로젝트 운영 매니저다. 해야 할 일을 명확하고 짧게 정리하라.",
        user_prompt=prompt,
        generation_name="tool-extract-action-items-llm",
    )
    return {"action_items_markdown": text}


@observe(name="generate_discussion_questions", as_type="tool")
def generate_discussion_questions(topic: str, audience: str = "대학생/청년") -> dict[str, Any]:
    prompt = f"""
    주제: {topic}
    청중: {audience}

    아래 형식으로 작성하라.
    1. 오프닝 질문 2개
    2. 핵심 토론 질문 5개
    3. 반대 관점 질문 3개
    4. 발표 후 청중 참여를 유도하는 마무리 질문 2개
    """
    text = llm_text(
        system_prompt="너는 대학 세미나 사회자 보조다. 정답보다 사고 확장을 유도하는 질문을 만든다.",
        user_prompt=prompt,
        generation_name="tool-generate-discussion-questions-llm",
    )
    return {"topic": topic, "audience": audience, "questions_markdown": text}


@observe(name="critique_presentation", as_type="tool")
def critique_presentation(draft: str) -> dict[str, Any]:
    prompt = f"""
    아래 발표 초안을 읽고 다음 형식으로 피드백하라.

    1. 핵심 주장 요약
    2. 논리적으로 약한 부분 3개
    3. 청중이 물을 가능성이 높은 질문 5개
    4. 먼저 수정하면 좋은 우선순위 3개
    5. 발표자가 마지막에 꼭 강조하면 좋은 한 줄

    [발표 초안]
    {draft}
    """
    text = llm_text(
        system_prompt="너는 발표 코치다. 칭찬보다 구조적 약점과 개선 우선순위를 선명하게 짚어라.",
        user_prompt=prompt,
        generation_name="tool-critique-presentation-llm",
    )
    return {"critique_markdown": text}


## 5. 함수 스키마 정의

모델은 아래 정의를 보고 어떤 함수를 쓸지 결정합니다.
즉, 이 부분이 OpenAI function calling의 핵심입니다.

In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "name": "summarize_notes",
        "description": "긴 회의 메모나 문서를 구조적으로 요약한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "notes": {
                    "type": "string",
                    "description": "회의 메모, 기사, 문서, 발표 준비 메모 등 원문 텍스트"
                }
            },
            "required": ["notes"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "extract_action_items",
        "description": "메모에서 해야 할 일을 우선순위와 함께 정리한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "notes": {
                    "type": "string",
                    "description": "회의 메모 또는 작업 노트 텍스트"
                }
            },
            "required": ["notes"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "generate_discussion_questions",
        "description": "세미나나 토론에서 쓸 질문을 생성한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "토론 주제"
                },
                "audience": {
                    "type": "string",
                    "description": "청중 특성"
                }
            },
            "required": ["topic", "audience"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "critique_presentation",
        "description": "발표 초안을 읽고 구조적 피드백과 예상 질문을 정리한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "draft": {
                    "type": "string",
                    "description": "발표 초안 또는 발표 스크립트"
                }
            },
            "required": ["draft"],
            "additionalProperties": False
        },
        "strict": True
    }
]

## 6. Agent 실행기

이제 실제 Agent 루프를 만듭니다.

Langfuse 관점에서 보면 이 함수 전체가 하나의 `agent` observation이 되고,
그 안에 모델 호출과 tool observation이 중첩되어 들어갑니다.

In [ ]:
def call_local_tool(name: str, args: dict[str, Any]) -> dict[str, Any]:
    if name == "summarize_notes":
        return summarize_notes(**args)
    if name == "extract_action_items":
        return extract_action_items(**args)
    if name == "generate_discussion_questions":
        return generate_discussion_questions(**args)
    if name == "critique_presentation":
        return critique_presentation(**args)
    raise ValueError(f"Unknown tool: {name}")


def print_tool_calls(response):
    for item in response.output:
        if getattr(item, "type", None) == "function_call":
            print(f"tool call -> {item.name}")
            print(f"arguments -> {item.arguments}")
            print("-" * 60)


@observe(name="run_openai_tool_agent", as_type="agent")
def run_openai_tool_agent(system_prompt: str, user_prompt: str, tools: list[dict[str, Any]], tool_choice: Any = "auto", run_name: str = "agent-run"):
    """
    OpenAI Responses API + function calling 기반 Agent 실행기.
    Langfuse Trace에서는 agent observation으로 기록됩니다.
    """
    response = openai.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tools=tools,
        tool_choice=tool_choice,
        name=run_name,
        metadata={"demo": run_name, "layer": "agent_entry"},
    )

    print_tool_calls(response)

    while True:
        function_calls = [item for item in response.output if getattr(item, "type", None) == "function_call"]
        if not function_calls:
            return response

        tool_outputs = []
        for tool_call in function_calls:
            args = json.loads(tool_call.arguments)
            result = call_local_tool(tool_call.name, args)
            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(result, ensure_ascii=False),
                }
            )

        response = openai.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
            name=f"{run_name}-followup",
            metadata={"demo": run_name, "layer": "agent_followup"},
        )


def show_text_response(response):
    print(response.output_text)


def flush_langfuse():
    """
    짧은 노트북 실행에서는 background 전송이 끝나기 전에 셀이 종료될 수 있어,
    시연 후 flush를 해주면 Trace가 UI에 더 빨리 보입니다.
    """
    try:
        langfuse.flush()
        print("Langfuse flushed")
    except Exception as exc:
        print("Langfuse flush skipped:", exc)

## 7. 실습 1: 회의 메모 정리 Agent

Trace에서 기대하는 흐름:

- `run_openai_tool_agent` (agent)
- `summarize_notes` (tool)
- `extract_action_items` (tool)
- 각 tool 내부의 LLM generation
- 최종 자연어 응답

In [ ]:
meeting_prompt = f"""
다음 회의 메모를 읽고, 발표 준비용으로 정리해줘.

요청사항:
1. 전체 내용을 짧고 밀도 있게 요약
2. 핵심 포인트 정리
3. 액션아이템 분리
4. 발표자가 꼭 강조해야 할 메시지 제안

[회의 메모]
{meeting_notes}
"""

with propagate_attributes(session_id=DEMO_SESSION_ID, user_id=DEMO_USER_ID, tags=["meeting-demo", "next-seodang"]):
    meeting_response = run_openai_tool_agent(
        system_prompt="너는 세미나 준비를 돕는 프로젝트 운영 보조자다. 문서 요약과 액션 구조화를 잘해야 한다.",
        user_prompt=meeting_prompt,
        tools=[tool for tool in TOOL_SCHEMAS if tool["name"] in {"summarize_notes", "extract_action_items"}],
        tool_choice="required",
        run_name="meeting-notes-agent",
    )

show_text_response(meeting_response)
flush_langfuse()

## 8. 실습 2: 토론 질문 생성 Agent

Trace에서 기대하는 흐름:

- `run_openai_tool_agent` (agent)
- `generate_discussion_questions` (tool)
- tool 내부 generation
- 최종 응답

In [ ]:
discussion_prompt = f"""
아래 토론 자료를 참고해서, 'AI Native 시대의 커리어'를 주제로 세미나 질문을 만들어줘.

청중은 대학생/청년이고,
질문은 생각을 넓히는 방향이어야 한다.

[참고 자료]
{discussion_material}
"""

with propagate_attributes(session_id=DEMO_SESSION_ID, user_id=DEMO_USER_ID, tags=["discussion-demo", "next-seodang"]):
    discussion_response = run_openai_tool_agent(
        system_prompt="너는 세미나 사회자 보조다. 정답보다 사고 확장을 유도하는 질문을 구성하라.",
        user_prompt=discussion_prompt,
        tools=[tool for tool in TOOL_SCHEMAS if tool["name"] == "generate_discussion_questions"],
        tool_choice="required",
        run_name="discussion-question-agent",
    )

show_text_response(discussion_response)
flush_langfuse()

## 9. 실습 3: 발표 피드백 Agent

Trace에서 기대하는 흐름:

- `run_openai_tool_agent` (agent)
- `critique_presentation` (tool)
- tool 내부 generation
- 최종 응답

In [ ]:
presentation_prompt = f"""
아래 발표 초안을 읽고, 발표 피드백을 해줘.

중점:
1. 논리적으로 약한 부분
2. 청중이 혼란스러워할 지점
3. 예상 질문
4. 먼저 고쳐야 할 우선순위

[발표 초안]
{presentation_draft}
"""

with propagate_attributes(session_id=DEMO_SESSION_ID, user_id=DEMO_USER_ID, tags=["presentation-demo", "next-seodang"]):
    presentation_response = run_openai_tool_agent(
        system_prompt="너는 발표 피드백 코치다. 과한 칭찬보다 구조적 보완 포인트를 먼저 짚어라.",
        user_prompt=presentation_prompt,
        tools=[tool for tool in TOOL_SCHEMAS if tool["name"] == "critique_presentation"],
        tool_choice="required",
        run_name="presentation-feedback-agent",
    )

show_text_response(presentation_response)
flush_langfuse()

## 10. 선택 확장: 여러 tool을 모델이 스스로 선택하게 하기

이 셀은 `tool_choice="auto"`를 써서,
모델이 어떤 함수를 쓸지 스스로 고르게 만드는 예시입니다.

Trace에서 보면 여러 tool이 연속 호출되는 모습을 확인할 수 있습니다.

In [ ]:
freeform_request = f"""
아래 세미나 준비 메모를 읽고,
- 먼저 내용을 정리하고
- 발표 준비를 위한 액션아이템을 뽑고
- 세션에서 쓸 토론 질문도 같이 제안해줘.

[메모]
{meeting_notes}
"""

with propagate_attributes(session_id=DEMO_SESSION_ID, user_id=DEMO_USER_ID, tags=["freeform-demo", "next-seodang"]):
    freeform_response = run_openai_tool_agent(
        system_prompt="너는 세미나 운영 보조 Agent다. 필요한 경우 여러 도구를 사용해도 된다.",
        user_prompt=freeform_request,
        tools=TOOL_SCHEMAS,
        tool_choice="auto",
        run_name="freeform-multi-tool-agent",
    )

show_text_response(freeform_response)
flush_langfuse()

## 11. Langfuse에서 어떻게 확인하면 좋은가

실행 후 Langfuse UI에서 아래를 보면 됩니다.

1. `session_id = next-seodang-agent-demo` 로 필터링
2. `meeting-notes-agent`, `discussion-question-agent`, `presentation-feedback-agent` trace 확인
3. 각 trace 안에서 tool observation 이름 확인
   - `summarize_notes`
   - `extract_action_items`
   - `generate_discussion_questions`
   - `critique_presentation`
4. tool 아래에 nested generation이 잡히는지 확인

즉, 발표 중에는 이렇게 말하면 됩니다.

- “모델이 어떤 함수를 골랐는지 지금 Trace에서 볼 수 있습니다.”
- “단순히 최종 답만 보는 게 아니라, 중간 과정까지 관찰할 수 있습니다.”

## 12. 핵심 정리

이 노트북에서 참가자가 가져가야 할 핵심은 다음입니다.

1. Agent는 거대한 시스템이 아니라, 함수 몇 개를 가진 작은 루프로도 시작할 수 있다.
2. OpenAI function calling을 쓰면 모델이 어떤 함수를 선택할지 판단할 수 있다.
3. Langfuse를 붙이면 `요청 -> 함수 선택 -> 함수 실행 -> 최종 응답`을 Trace로 볼 수 있다.
4. 품질이 중요할 때는 도구 함수 내부에서도 LLM을 다시 쓰는 현실적 예외를 설계할 수 있다.
5. 결국 중요한 것은 좋은 도구 설계와 검수 가능한 운영 방식이다.

## 참고 문서

OpenAI 공식 문서:
- [Quickstart (Responses API, Python)](https://platform.openai.com/docs/quickstart?api-mode=responses&lang=python)
- [Function Calling Guide](https://platform.openai.com/docs/guides/function-calling?api-mode=responses&lang=python)
- [gpt-4.1-mini 모델 정보](https://platform.openai.com/docs/models/gpt-4.1-mini)

Langfuse 공식 문서:
- [Get Started](https://langfuse.com/docs/observability/get-started)
- [OpenAI Integration](https://langfuse.com/docs/integrations/openai)
- [Langfuse Python SDK Overview](https://langfuse.com/docs/observability/sdk/overview)

강의용 보조 파일:
- `docs/lecture/ppt_detailed_script_60p.md`
- `examples/data/seminar_meeting_notes_long.md`
- `examples/data/discussion_material_ai_native.md`
- `examples/data/presentation_draft_ai_native.md`